# Introdução

Baseado em: https://www.kaggle.com/alexisbcook/machine-learning-competitions).


## Imports

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV
import matplotlib.pyplot as plt
%matplotlib inline

## Dataset: House Prices

In [ ]:
df = pd.read_csv('../input/train.csv')
y = df.SalePrice
df.describe()

In [ ]:
df.info()

In [ ]:
# Escolhendo alguns atributos
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'OverallQual', 'OverallCond', 'GarageArea', 'ExterQual', 'ExterCond'] #'OverallQual', 'OverallCond'

X = df[features]
#X = df.drop(columns = "SalePrice")
#X = df.dropna(axis=1, how='any')
#X = df.fillna(df.mean())
X.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(drop='first',sparse_output=False)
X_enc = enc.fit_transform(X[['ExterQual', 'ExterCond']])

In [ ]:
X_enc

In [ ]:
encoded_df = pd.DataFrame(X_enc, columns=enc.get_feature_names_out(['ExterQual', 'ExterCond']))


In [ ]:
encoded_df

In [ ]:
df_transformed = pd.concat([X.drop(columns=['ExterQual', 'ExterCond']), encoded_df], axis=1)

In [ ]:
X = df_transformed

In [ ]:
X

In [ ]:
# Dados de treino e validação
x_train, x_val, y_train, y_val = train_test_split(X, y, random_state=1, test_size = 0.2)
print("Qtde train: "+ str(len(x_train)))
print("Qtde val: "+ str(len(x_val)))

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scale = StandardScaler()
x_train = scale.fit_transform(x_train)
x_val = scale.transform(x_val)

## Treinando um Modelo

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100)
rf.fit(x_train, y_train)
y_pred = rf.predict(x_val)
rf_val_mae = mean_absolute_error(y_pred, y_val)

print("Validation MAE for Random Forest Model: {:,.0f}".format(rf_val_mae))

In [ ]:
plt.plot(y_val-y_pred, 'o')

## Estimando Dados de Teste

In [ ]:
dft = pd.read_csv('../input/test.csv')
x_test = dft[features]
x_test.head()

In [ ]:
x_test.info()

In [ ]:
x_test = x_test.fillna(x_test.mean())

In [ ]:
y_test = rf.predict(x_test)

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(objective='reg:squarederror')

# Definir a grade de parâmetros para a busca
param_grid = {
    'colsample_bytree': [0.3, 0.7],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 5, 7],
    'alpha': [0, 10, 50],
    'n_estimators': [50, 100, 200]
}

# Configurar o GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, 
                           scoring='neg_mean_squared_error', cv=3, verbose=1)

# Executar a busca pelos melhores parâmetros
grid_search.fit(x_train, y_train)

# Exibir os melhores parâmetros encontrados
print(f"Melhores parâmetros: {grid_search.best_params_}")

# Treinar o modelo com os melhores parâmetros
best_model = grid_search.best_estimator_

# Fazer previsões com o modelo otimizado
y_pred = best_model.predict(x_val)

rf_val_mae = mean_absolute_error(y_pred, y_val)

print("Validation MAE for Random Forest Model: {:,.0f}".format(rf_val_mae))

# Gerando uma submissão

In [ ]:
# Run the code to save predictions in the format used for competition scoring
output = pd.DataFrame({'Id': dft.Id,
                       'SalePrice': y_test})
output.to_csv('submission.csv', index=False)

## Submissão da competição

* Entra na competição (https://www.kaggle.com/c/home-data-for-ml-course).
* Salva notebook executando tudo
* Envia o arquivo submission.csv

![join competition image](https://i.imgur.com/axBzctl.png)

## Desafio

* Usar mais atributos
* Limpar dataset (cuidado com atributos nulos)
* Escalar atributos
* Criar novos atributos
* Usar modelos melhores (ex. XGBoost)


---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intro-to-machine-learning/discussion) to chat with other learners.*